Makemore NLP Language Model

This project is similar to the previous in the sense that its using the same data set and has the same task of predicting the next character. However, instead of only taking the previous character as context for predicting the next, we will increase the context length to N (typically N=3). Additionally, the architecture of the model will be slightly more complicated, instead of a single matmul we will design a Multi-Layer Perceptron Model


Step 1: Imports, read names.txt, and build chars,stoi,itos (as previously done)

In [18]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print(len(words), words[:5])
print(stoi)
print(itos[5])
print(chars[:5])

32033 ['emma', 'olivia', 'ava', 'isabella', 'sophia']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
e
['a', 'b', 'c', 'd', 'e']


Previously the dataset was a list of pairs (prev_char, next_char). Now its a list of pairs (Prev_N_chars, next_char), where N= block_size. Lets start .emma. as an example. (..., e) -> (..e,m) -> (.em, m) -> (emm,a) -> (mma, .). Notice, since the word length is 4, for a couple of the examples we needed to pad with '.'. Now lets create this dataset

In [21]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

print(X.shape, X.dtype)
print(Y.shape, Y.dtype)
print(X[:10])
print(Y[:10])

for x,y in zip(X[:20], Y[:20]):
     print(''.join(itos[i.item()] for i in x), '-->', itos[y.item()])


torch.Size([228146, 3]) torch.int64
torch.Size([228146]) torch.int64
tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22]])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
... --> a
..a --> v
.av --> a
ava --> .
... --> i
..i --> s
.is --> a
isa --> b


Step 2: Create the lookup table (embedding table).

We want an embedding for each of the 27 characters. Therefore, we will have 27 rows. For now, lets assume each embedding is 2-Dimensional. Therefore, the lookup table will have shape (27,2). Initially, these will be random numbers

In [22]:
C=torch.randn((27,2))
emb = C[X]

print(C.shape)
print(emb.shape)
print(emb[0])
print(emb[0,0])
print(C[X[0,0]])

torch.Size([27, 2])
torch.Size([228146, 3, 2])
tensor([[-1.4916,  0.6273],
        [-1.4916,  0.6273],
        [-1.4916,  0.6273]])
tensor([-1.4916,  0.6273])
tensor([-1.4916,  0.6273])


The lookup table has a 2-Dimensional embedding for each of the characters. Therefore the embedding for one character has the shape (2,). If we take a single data point which has 3 characters, each character has its own vector embedding hence the shape of the vector embedding for one data point becomes (3,2). Therefore for the whole dataset of N examples, all these examples are stacked to create a shape (N,3,2)

Step 3: Flatten + Hidden Layer

After the lookup table, the next layer of the MLP is the hidden layer (w1x1 + w2x2 + w3x3.... wNxN + bias). However, the issue that we have now is that our input is 3-Dimensional (N,3,2). Which means, before applying the weights (matmul), we need to flatten it into a 2-Dimensional input